# Tiêu chí 4 · Tính nguyên bản

Tính tỷ lệ ảnh có watermark. Điểm nguyên bản do người chấm lấy từ observations.

Đóng gói lượt của tiêu chí theo [hướng dẫn Colab](../README.md) rồi đặt đường dẫn ZIP trong ô đầu.

In [ ]:
from pathlib import Path
from google.colab import drive

drive.mount("/content/drive")
CRITERION_ID = "4"
CRITERION_NAME = "originality"
INPUT_ZIP = Path("/content/drive/MyDrive/HELM/originality-colab-input.zip")
if not INPUT_ZIP.is_file():
    raise FileNotFoundError(f"Không tìm thấy ZIP trên Drive: {INPUT_ZIP}")

## Giải nén và kiểm tra đầu vào

In [ ]:
import json
import re
import sqlite3
import stat
from contextlib import closing
from pathlib import PurePosixPath
from tempfile import mkdtemp
from zipfile import ZipFile

WORK_DIR = Path(mkdtemp(prefix="helm-demo-"))
with ZipFile(INPUT_ZIP) as archive:
    for entry in archive.infolist():
        member = PurePosixPath(entry.filename)
        if member.is_absolute() or ".." in member.parts or stat.S_ISLNK(entry.external_attr >> 16):
            raise ValueError(f"Đường dẫn ZIP không hợp lệ: {entry.filename}")
    archive.extractall(WORK_DIR)

manifest = json.loads((WORK_DIR / "evaluation.json").read_text(encoding="utf-8"))
run = manifest["run"]
if not re.fullmatch(r"[A-Za-z0-9][A-Za-z0-9_.-]{0,79}", run):
    raise ValueError("Tên lượt trong evaluation.json không hợp lệ.")
if CRITERION_ID not in manifest["criteria"]:
    raise ValueError(f"ZIP không có dữ liệu cho tiêu chí {CRITERION_ID} ({CRITERION_NAME}).")
db_relative = Path("results/benchmark") / run / "requests.sqlite3"
db_path = WORK_DIR / db_relative
if "database" in manifest and manifest["database"] != db_relative.as_posix():
    raise ValueError("Đường dẫn SQLite trong evaluation.json không khớp với lượt chạy.")
if not db_path.is_file():
    legacy_db = WORK_DIR / "outputs/benchmark/runs" / run / "requests.sqlite3"
    if not legacy_db.is_file():
        raise FileNotFoundError(f"ZIP thiếu SQLite: {db_relative}")
    db_path.parent.mkdir(parents=True, exist_ok=True)
    legacy_db.replace(db_path)

image_counts = manifest.get("image_counts")
if image_counts is None:
    with closing(sqlite3.connect(db_path.as_uri() + "?mode=ro", uri=True)) as connection:
        image_counts = dict(connection.execute("""
            SELECT criterion, COUNT(*) FROM results
            WHERE json_extract(record, '$.status') = 'success'
            GROUP BY criterion
        """))
image_count = image_counts.get(CRITERION_NAME, 0)
print(f"Lượt chạy {run} · {CRITERION_NAME}: {image_count:,} ảnh")

In [ ]:
import subprocess
import sys

if image_count:
    subprocess.run([
        sys.executable, "-m", "pip", "install",
        "Pillow",
        "crfm-helm==0.5.16",
        "timm==0.6.13",
    ], check=True)

## Đánh giá, hiện kết quả và xuất ZIP

Ô cuối gọi `evaluate` để chấm ảnh đầu ra, hiện kết quả từng metric và lưu ZIP báo cáo cạnh ZIP đầu vào trên Google Drive.

In [ ]:
import subprocess
import sys
from html import escape
from IPython.display import HTML, display
from tempfile import NamedTemporaryFile
from zipfile import ZIP_DEFLATED

command = [sys.executable, "benchmark.py", "evaluate", "--run", run, "--criteria", CRITERION_ID]
if manifest["observations"]:
    command.extend(["--observations", "observations.jsonl"])
subprocess.run(command, cwd=WORK_DIR, check=True)

summary_relative = Path("results/benchmark") / run / "summary.json"
summary_path = WORK_DIR / summary_relative
summary = json.loads(summary_path.read_text(encoding="utf-8"))
archive_prefix = INPUT_ZIP.name.removesuffix("-colab-input.zip") if INPUT_ZIP.name.endswith("-colab-input.zip") else run
output_zip = INPUT_ZIP.with_name(f"{archive_prefix}-{CRITERION_NAME}-colab-results.zip")
with NamedTemporaryFile(dir=output_zip.parent, suffix=".zip", delete=False) as temporary:
    temporary_path = Path(temporary.name)
try:
    with ZipFile(temporary_path, "w", compression=ZIP_DEFLATED) as archive:
        archive.write(summary_path, summary_relative.as_posix())
    temporary_path.replace(output_zip)
finally:
    temporary_path.unlink(missing_ok=True)

rows = []
for name, criterion in summary["criteria"].items():
    for metric, data in criterion["metrics"].items():
        value = data.get("value")
        coverage = f"{data['n']}/{data['expected']}" if "expected" in data else str(data.get("n", "—"))
        cells = (name, metric, f"{value:.6g}" if value is not None else "—", coverage, data.get("error", ""))
        rows.append("<tr>" + "".join(f"<td>{escape(cell)}</td>" for cell in cells) + "</tr>")
counts = "; ".join(
    f"{name}: {criterion['samples']} mẫu · {criterion['by_status']}"
    for name, criterion in summary["criteria"].items()
)
display(HTML(
    f"<p>{escape(counts)}</p>"
    "<table><thead><tr><th>Tiêu chí</th><th>Metric</th><th>Điểm</th><th>Số mẫu</th><th>Ghi chú</th></tr></thead>"
    "<tbody>" + "".join(rows) + "</tbody></table>"
))
print(f"ZIP kết quả trên Google Drive: {output_zip}")